---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: c:\Users\iarin\Desktop\inginerie AI\echochamber-project-team-6
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [4]:
import pandas as pd
import random

corpus = pd.read_json("../../data/cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[RecorderRomania] Oameni pentru care lasi tot si iesi in strada sa ii sustii ❤ Va multumim pentru 
[NicusorDanRO] M-a cam dezamagit președintele Dan în ultimul timp în special în ceea ce priveșt
[@CălinGeorgescu-CanalulOficial] Trăiască președintele ales .Cel mai bun in a conduce frumoasa tara România va iu


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [5]:
# modifica dupa preferinte

AXA_1 = "geopolitic"
AXA_2 = "religious_frame"
AXA_3 = "fear"
AXA_4 = "conspiracy"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [6]:
AXA_1_DEFINITION = """
geopolitic măsoară dacă textul interpretează evenimentele politice în termeni de putere, teritoriu, influență între state sau actori globali.
0 = absent
1 = prezent
2 = dominant
"""
AXA_2_DEFINITION = """
religious_frame măsoară dacă textul interpretează evenimentele în termeni de credință, divinitate, mântuire, sau alte concepte religioase.
0 = absent
1 = prezent
2 = dominant
"""
AXA_3_DEFINITION = """
fear măsoară dacă textul încearcă să inducă frică, anxietate sau panică în cititor.
0 = absent
1 = prezent
2 = dominant
"""
AXA_4_DEFINITION = """
conspiracy măsoară dacă textul promovează sau susține teorii ale conspirației, adică explicații alternative care implică acțiuni secrete sau intenții malefice din partea unor grupuri sau indivizi.
0 = absent
1 = prezent
2 = dominant
""" 


## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [7]:
MINI_PROMPT = f"""
Ești un expert în analiza discursului politic, specializat în identificarea temelor și tonurilor din comentariile online. Ai fost angajat să adnotezi un comentariu politic folosind patru axe specifice: {AXA_1}, {AXA_2}, {AXA_3} și {AXA_4}. Fiecare axă are o definiție clară și un set de reguli pentru codare.
SARCINĂ:
Adnotează comentariul folosind patru axe:
1. {AXA_1}
2. {AXA_2}
3. {AXA_3}
4. {AXA_4}
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2
{AXA_3} = 0 / 1 / 2
{AXA_4} = 0 / 1 / 2
DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}
{AXA_3_DEFINITION}
{AXA_4_DEFINITION}
REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0,
  "{AXA_3}": 0,
  "{AXA_4}": 0
}}
"""
print(MINI_PROMPT)


Ești un expert în analiza discursului politic, specializat în identificarea temelor și tonurilor din comentariile online. Ai fost angajat să adnotezi un comentariu politic folosind patru axe specifice: geopolitic, religious_frame, fear și conspiracy. Fiecare axă are o definiție clară și un set de reguli pentru codare.
SARCINĂ:
Adnotează comentariul folosind patru axe:
1. geopolitic
2. religious_frame
3. fear
4. conspiracy
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
geopolitic = 0 / 1 / 2
religious_frame = 0 / 1 / 2
fear = 0 / 1 / 2
conspiracy = 0 / 1 / 2
DEFINIȚII:

geopolitic măsoară dacă textul interpretează evenimentele politice în termeni de putere, teritoriu, influență între state sau actori globali.
0 = absent
1 = prezent
2 = dominant


religious_frame măsoară dacă textul interpretează evenimentele 

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [8]:
TESTS = corpus.sample(10, random_state=42)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
145,yt_BfvZ8QcVBKc_Ugyog0iMEqAX5zIQb4R4AaABAg,turcescu111,Harpalete- Sângerete și transfuzia din lumea lui,"Da, si eu cred ca serviciile ucrainene au fost..."
334,yt_qkGhsJFft00_UgzJHKCoGkwtTJ3uOBh4AaABAg,digi24hd56,În fața ta cu Emil Hurezeanu: „Ar fi un coșmar...,Ce mă supără pe mine oamenii ăștia care sunt a...
175,yt_KqrUotq1Obs_Ugy6tYmdfH0T2THArnB4AaABAg,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pace și prosperitate ( 28.10...,Bunul Dumnezeu să îl protejeze pe președintele...
369,yt_vkP6FdP9iX0_Ugwj3HojTt6JikJVtjd4AaABAg,turcescu111,Orientul Mijlociu în flăcări,Nicușor merge pe lângă covor pentru că nu are ...
416,yt_Sj4fQKlMOro_UgyNWIwNDtUeTTCrLkl4AaABAg,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Trebuie susținută aceasta femeie!!!!! Acesta a...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [12]:
USE_GEMINI = False
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: deepseek-chat


In [13]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [14]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Da, si eu cred ca serviciile ucrainene au fost inplicate in alegerile noastre. Am convingerea ca serviciile noastre sunt infiltrate de serviciile ucrainene

OUTPUT MODEL:
```json
{
  "target": "serviciile ucrainene",
  "stance": "anti",
  "tone": "acuzator",
  "geopolitic": 1,
  "religious_frame": 0,
  "fear": 1,
  "conspiracy": 2
}
```
COMENTARIU:
Ce mă supără pe mine oamenii ăștia care sunt așa de siguri când spun ca Iranul nu mai are arsenal militar. De unde mama dracului știu ei treburile astea. Noi bănuim ca primesc arme din Rusia bolshevica și China comunistă. Poate și Brazilia dar iarăși e o bănuială.

OUTPUT MODEL:
{
  "target": "Iran",
  "stance": "anti",
  "tone": "acuzator",
  "geopolitic": 2,
  "religious_frame": 0,
  "fear": 1,
  "conspiracy": 2
}
COMENTARIU:
Bunul Dumnezeu să îl protejeze pe președintele nostru Călin Georgescu ❤️❤️❤️

OUTPUT MODEL:
{
  "target": "Călin Georgescu",
  "stance": "pro",
  "tone": "afectiv",
  "geopolitic": 0,
  "religious_frame": 

In [ ]:
## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce axe ai ales?
- De ce le-ai ales?
- Modelul a returnat JSON corect?
- Care a fost cea mai mare problemă?
- Ce ai schimba în prompt?

Ce axe ai ales?
geopolitic, religious_frame, fear, conspiracy — două din curs, două proprii.
De ce le-ai ales?
Combinate, detectează un tip specific de discurs populist-naționalist frecvent în comentariile politice românești de pe YouTube.
Modelul a returnat JSON corect?
Structura a fost respectată în toate cele 10 cazuri, dar câteva răspunsuri aveau text introductiv înainte de JSON — necesitând curățare cu regex.
Care a fost cea mai mare problemă?
Suprapunerea fear ↔ conspiracy: comentariile conspiraționiste induc aproape mereu și frică, deci modelul a codat adesea ambele ca 2 = dominant, reducând discriminarea.
Ce ai schimba în prompt?
O regulă de prioritizare între axele suprapuse + 1–2 exemple few-shot cu comentarii deja adnotate, pentru a ancora mai bine scala 0/1/2.